In [ ]:
import os
import wfdb
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

In [ ]:
'''
supin_transition = [
    'Conclude slow tilt down',
    'Transition back to supine',
    'End slow tilt down',
    'End fast tilt down',
    'Back to supine',
    'End rapid tilt down',
    'Conclusion of rapid tilt down',
    'Conclude rapid tilt down',
    'Conclusion of slow tilt down',
    'Conclusion of slow tilt down',
    'Conclusion of rapid tilt dow',
    'Transition back to supin',
    'Conclude slow tilt dow',
    'Conslusion of slow tilt down',
    '   Conclusion of slow tilt down','Conslusion of slow tilt down'
]
'''

supin_transition = [
    'Transition back to supine',
    'Back to supine',
    'Transition back to supin'
]

stand_up_transition = [
    'Stand up','Stand-up from supine','0 -1 0 \tStand up','Stand-up']

In [ ]:
# ==========================================
# 1. LETTURA DATASET E PREPARAZIONE DATI 
# ==========================================

CURRENT_PATH = os.getcwd()
ID_TILT_TEST_PATH = '/Users/alessiorizzi/Desktop/env_tesi/dataset/supin_stand/RECORDS.txt'
DATA_DIR = '/Users/alessiorizzi/Desktop/env_tesi/dataset/supin_stand/'

with open (ID_TILT_TEST_PATH) as file:
  id_pazienti = file.readlines()

id_pazienti = [x.strip() for x in id_pazienti]

patien_dict = {id_p: [] for id_p in id_pazienti}

for id_p in id_pazienti:

  record_full_path = os.path.join(DATA_DIR, id_p)
  record = wfdb.rdrecord(record_full_path)
  annotation = wfdb.rdann(record_full_path, 'anI')

  signal_250Hz = record.p_signal[:, 1]      # Il segnale ECG vero e proprio (matrice numpy)
  fs = record.fs                            # Frequenza di campionamento 250
  leads = record.sig_name                   # ['ABP', 'ECG', 'Angle']


  signal_100Hz = signal.resample_poly(signal_250Hz, up=2, down=5)
  transition = annotation.aux_note          # nomi delle transizioni
  samples = np.array(np.round(annotation.sample / 2.5).astype(int), dtype='int') # numero di sample per ECG

  for i, (name_tr, sample) in enumerate(zip(transition, samples)):

    # 1. Viene identificata la label (0, 1 o None se non ci interessa)
    if name_tr in supin_transition:
        label = 0
    elif name_tr in stand_up_transition:
        label = 1
    else:
        continue  # Salta le transizioni che non sono nelle due liste

    # 2. Se siamo all'ultimo indice, prendiamo dal sample attuale fino alla fine
    if i == len(transition) - 1:
        ecg = signal_100Hz[sample:]
    else:
      # Altrimenti prendiamo dall'inizio (0) o dal sample precedente fino a quello attuale
      start = 0 if i == 0 else samples[i - 1]
      ecg = signal_100Hz[start:sample]

    # 3. Aggiunta al dizionario
    if len(ecg) > 344:
      patien_dict[id_p].append((ecg, label))

In [ ]:
# ==========================================
# 2. CREAZIONE DIZIONARIO 
# ==========================================

nuovo_dict = {}
chunk_size = 344

for paziente_id, registrazioni in patien_dict.items():
    nuovo_dict[paziente_id] = []

    for ecg_array, label in registrazioni:
        # Calcoliamo quanti blocchi da 1000 possiamo estrarre
        n_chunks = len(ecg_array) // chunk_size

        for i in range(n_chunks):
            start = i * chunk_size
            end = start + chunk_size

            # Estraiamo il segmento
            segmento = ecg_array[start:end]

            # Aggiungiamo alla lista del paziente come tupla (segmento, label)
            nuovo_dict[paziente_id].append((segmento, label))


In [ ]:
# ==========================================
# 3. OGNI ECG D'INTERESSE VIENE SUDDIVISO IN 
#    ECG DI DIMENSIONE 344 
# ==========================================

ORIGINAL_ECG_DIR = '/Users/alessiorizzi/Desktop/env_tesi/img/tot_ecg'

if not os.path.exists(ORIGINAL_ECG_DIR):
    os.makedirs(ORIGINAL_ECG_DIR)

for paziente_id, segmenti in nuovo_dict.items():

    # 1. Crea la cartella del paziente
    path_paziente = os.path.join(ORIGINAL_ECG_DIR, paziente_id)

    # 2. Crea le sottocartelle 0 e 1 per le label
    for label_dir in ['0', '1']:
        os.makedirs(os.path.join(path_paziente, label_dir), exist_ok=True)

    print(f"Elaborazione Paziente {paziente_id}...")

    for idx, (ecg_array, label) in enumerate(segmenti):
        # Definiamo il percorso finale: dataset_ecg_visual/ID/LABEL/IDX.png
        nome_file = f"{idx}.png"
        path_salvataggio = os.path.join(path_paziente, str(label), nome_file)

        # 3. Generazione del grafico
        plt.figure(figsize=(10, 4))
        plt.plot(ecg_array, color='blue', linewidth=0.5)

        # 4. Salvataggio
        plt.savefig(path_salvataggio, bbox_inches='tight', pad_inches=0)
        plt.close()  # Chiude la figura per liberare memoria RAM

print("Salvataggio completato!")

In [ ]:
import matplotlib.pyplot as plt

# Carica i segnali
# p_signal[:, 0] -> ECG
# p_signal[:, 1] -> Gradi del lettino (se presente come canale)
ecg_signal = record.p_signal[:, 1]
tilt_angle = record.p_signal[:, 2] # misurazione in gradi del lettino

# Se hai fatto il resample a 100Hz per l'ECG,
# devi farlo identico anche per il segnale dell'inclinazione!

# Visualizzazione
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Grafico ECG
ax1.plot(signal_100Hz[58828:63841], color='blue')
ax1.set_ylabel('ECG (mV)')
ax1.set_title('Monitoraggio ECG e Inclinazione Lettino')

# Grafico Inclinazione
ax2.plot(tilt_angle[58828:63841], color='red')
ax2.set_ylabel('Inclinazione (Gradi °)')
ax2.set_xlabel('Campioni (100Hz)')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 4. CARICAMENTO DEI SOLI SEGNALI ECG NON DISTURBATI 
# ==========================================

from pathlib import Path

PATH = '/Users/alessiorizzi/Desktop/env_tesi/img/filtrati_ecg'
percorso = Path(PATH)
labels = ['0', '1']

test_dict = {}

for key in nuovo_dict.keys():

    indici_paziente = []

    for label in labels:
        PATH_COMPLETO = percorso / key / label
        if PATH_COMPLETO.exists():
            # Prendiamo gli indici dei file presenti nella cartella di QUESTO paziente
            nomi_file = [int(f.name.split('.png')[0]) for f in PATH_COMPLETO.iterdir() if f.is_file()]
            indici_paziente.extend(nomi_file)

    # Trasformiamo in set per velocità di ricerca
    set_indici = set(indici_paziente)

    # --- FILTRAGGIO IMMEDIATO per questo paziente ---
    # Prendiamo i dati da nuovo_dict[key] solo se l'indice i esiste tra i file PNG
    dati_originali = nuovo_dict[key]
    test_dict[key] = [dati_originali[i] for i in range(len(dati_originali)) if i in set_indici]

    print(f"Paziente {key}: trovati {len(indici_paziente)} file, filtrati {len(test_dict[key])} elementi.")

# A questo punto test_dict è il tuo nuovo dizionario filtrato correttamente

In [ ]:
# ==========================================
# 5. NORMALIZZAZIONE E LINEAR_DECAY PER TRAINING
# ==========================================

def z_score(X, mean=None, std=None):
    # se non passiamo mean e std li calcoliamo, caso X_train
    if mean is None or std is None:
        mean = np.mean(X)
        std = np.std(X)

    X_normalized = (X - mean) / (std + 1e-7)

    return X_normalized, mean, std


# Fine-Tuning linear decay
def linear_decay_end(epoch):

    initial_lr = 1e-3   # inizialmente era 1e-2 (+1)
    final_lr = 1e-5     # inizialmente era 1e-4 (-1)
    epochs = 100

    # Formula del decadimento lineare
    lr = initial_lr - (initial_lr - final_lr) * (epoch / epochs)

    return float(lr)

In [ ]:
# ==========================================
# 6. TECNICHE DI DATA AUGMENTATION (DA)
# ==========================================

import tensorflow as tf
from tensorflow.keras import layers

import seaborn as sns
from sklearn.metrics import confusion_matrix

def augment_ecg(batch_x, batch_y):
    """
    Applica Gaussian Noise e Amplitude Scaling on-the-fly.
    batch_x shape: (batch_size, 344, 1)
    """

    # 1. AMPLITUDE SCALING [0.7, 1.3]
    # Generiamo un fattore casuale per ogni elemento del batch
    alpha = tf.random.uniform(shape=[tf.shape(batch_x)[0], 1, 1], minval=0.7, maxval=1.3)
    batch_x = batch_x * alpha

    # 2. GAUSSIAN NOISE (std dev in [0.01, 0.1])
    # Scegliamo una deviazione standard casuale per il rumore
    std_dev = tf.random.uniform(shape=[], minval=0.01, maxval=0.1)
    noise = tf.random.normal(shape=tf.shape(batch_x), mean=0.0, stddev=std_dev)
    batch_x = batch_x + noise

    # Nota sul Time Stretching:
    # Scalare temporalmente un segnale fisso a 344 campioni è complesso on-the-fly
    # perché cambierebbe la lunghezza. Di solito per i battiti centrati ci si
    # ferma a Noise e Amplitude, che sono i più efficaci.

    return batch_x, batch_y

In [ ]:
# ==========================================
# 7. ARCHITETTURA CNN 
# ==========================================

def model_ptbxl(input_shape=(344, 1)):

    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(1, 1, kernel_initializer='ones', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    filters = [16, 16, 32, 32, 64, 64]
    kernels = [5, 5, 5, 3, 3, 3]

    # Temporal Fusion
    for i, (f, k) in enumerate(zip(filters, kernels)):
        x = layers.Conv1D(f, k, dilation_rate=2, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU()(x)
        x = layers.MaxPooling1D(2)(x)

    # Inter-lead Fusion
    x = layers.Conv1D(128, 1, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    # x = layers.Dropout(0.4)(x) # Protezione finale prima dell'output

    x = layers.Flatten()(x)
    #outputs = layers.Dense(5, activation='sigmoid')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)

    return model

def linear_decay(epoch):

    initial_lr = 1e-2   # inizialmente era 1e-2 (+1)
    final_lr = 1e-4     # inizialmente era 1e-4 (-1)
    epochs = 200

    # Formula del decadimento lineare
    lr = initial_lr - (initial_lr - final_lr) * (epoch / epochs)

    return float(lr)

In [ ]:
# ==========================================
# 8. FINE-TUNING E VALUTAZIONE (CROSS-VALIDATION)
# ==========================================

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

ECG_CM = './img/confusion_matrix/test_2'
ECG_METRICHE = './img/metriche/test_2'
MODEL_PRE_TRAINING = './model/best_ptbxl_d1_model_NORM_vs_ALL_344.keras'

if not os.path.exists(ECG_CM):
    os.makedirs(ECG_CM)

if not os.path.exists(ECG_METRICHE):
    os.makedirs(ECG_METRICHE)

keys = list(test_dict.keys())

for fold in range(len(keys)):
    X_test_temp, y_test_temp = [], []
    X_train_temp, y_train_temp = [], []

    key_test = [keys[fold]]
    keys_train = [x for j, x in enumerate(keys) if j != fold]

    X_train_dict = {k: test_dict[k] for k in keys_train}
    X_test_dict  = {k: test_dict[k] for k in key_test}

    for key, value in X_test_dict.items():
        for ecg, label in value:
            X_test_temp.append(ecg)
            y_test_temp.append(label)

    X_test_temp = np.array(X_test_temp, dtype='float32')
    y_test = np.array(y_test_temp)

    for key, value in X_train_dict.items():
        for ecg, label in value:
            X_train_temp.append(ecg)
            y_train_temp.append(label)

    X_train_temp = np.array(X_train_temp,dtype='float32')
    y_train = np.array(y_train_temp)

    X_train, mean, std = z_score(X_train_temp)
    X_test, _, _ = z_score(X_test_temp, mean, std)

    X_train = np.expand_dims(X_train, axis=-1)
    X_test = np.expand_dims(X_test, axis=-1)


    print(f"X_train: {X_train.shape}")
    print(f"X_test: {X_test.shape}")

    # Creazione del Dataset
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))

    # Pipeline: Shuffle -> Batch -> Augment
    train_ds = (train_ds
                .shuffle(buffer_size=X_train.shape[0])
                .batch(64) # o il tuo batch_size preferito
                .map(augment_ecg, num_parallel_calls=tf.data.AUTOTUNE)
                .prefetch(tf.data.AUTOTUNE))

    # Dataset di validazione (SENZA augmentation!)
    val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(64).prefetch(tf.data.AUTOTUNE)

    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-10) # Il valore iniziale

    old_model = tf.keras.models.load_model(MODEL_PRE_TRAINING, compile=False)
    x = old_model.layers[-2].output

    # 3.
    new_outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    # 4.
    binary_model = tf.keras.Model(inputs=old_model.input, outputs=new_outputs)
    for layer in binary_model.layers[:-1]:
        layer.trainable = False

    binary_model.compile(
        # optimizer = tf.keras.optimizers.SGD(learning_rate=1e-3, momentum=0.9),
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )


    # Assicurati che train_ds e val_ds siano pronti come abbiamo visto prima
    history = binary_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10
    )

    for layer in binary_model.layers:
        layer.trainable = True

    # IMPORTANTE: Usa un Learning Rate molto più basso!
    binary_model.compile(
        # optimizer = tf.keras.optimizers.SGD(learning_rate=1e-5, momentum=0.9),
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    checkpoint_path = f"best_model_fold_{fold}.h5"

    # Callback 1: Early Stopping
    # Interrompe se la val_loss non migliora per 20 epoche (patience)
    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

    # Callback 2: Model Checkpoint
    # Salva solo il modello che ha ottenuto la miglior val_loss finora
    checkpoint = ModelCheckpoint(filepath=checkpoint_path,
                                monitor='val_loss',
                                save_best_only=True,
                                verbose=1)

    lr_scheduler = tf.keras.callbacks.LearningRateScheduler(linear_decay_end)

    history_finetune = binary_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=100,
        callbacks=[lr_scheduler, checkpoint, early_stop]
    )


    # --- CALCOLO CONFUSION MATRIX CON IL MIGLIOR MODELLO ---

    # Carichiamo il modello migliore salvato dal checkpoint
    # (Anche se restore_best_weights=True lo fa in memoria, ricaricare dal file è più sicuro)
    best_model = tf.keras.models.load_model(checkpoint_path)

    # Eseguiamo la predizione sul set di test del fold
    y_pred_probs = best_model.predict(X_test)
    # --- Ottimizzazione automatica del Threshold ---
    from sklearn.metrics import roc_curve, f1_score

    fpr, tpr, thresholds = roc_curve(y_test, y_pred_probs)
    # Calcoliamo Youden Index
    best_threshold = thresholds[np.argmax(tpr - fpr)]

    # Se il threshold è > 1 o < 0 (capita con roc_curve), lo limitiamo
    best_threshold = np.clip(best_threshold, 0.05, 0.95)

    print(f"Fold {fold}: Best Threshold found = {best_threshold:.4f}")

    y_pred = (y_pred_probs >= best_threshold).astype(int)

    # Calcolo Matrice di Confusione con soglia dinamica
    cm = confusion_matrix(y_test, y_pred)

    # Plot e Salvataggio (come visto prima)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Supin', 'Stand-up'],
                yticklabels=['Supin', 'Stand-up'])
    plt.title(f'Confusion Matrix (Best Model) - Fold {fold}')
    plt.ylabel('Reale')
    plt.xlabel('Predetto')

    save_path = os.path.join(ECG_CM, f'confusion_matrix_fold_{fold}.png')
    plt.savefig(save_path)
    plt.close()

    # Opzionale: Rimuovi il file .h5 per non intasare lo spazio disco
    os.remove(checkpoint_path)

    ### AGGIUNTA

    print(f"Matrice di confusione salvata per il fold {fold} in: {save_path}")

    # Creiamo una figura con 2 grafici affiancati (1 riga, 2 colonne)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Training & Validation Metrics - Fold {fold}', fontsize=16, fontweight='bold')

    # Determiniamo il punto di separazione tra Feature Extraction e Fine-Tuning
    split_point = len(history.history['accuracy'])

    # --- 1. GRAFICO ACCURACY ---
    # Uniamo i dati delle due fasi
    total_accuracy = history.history['accuracy'] + history_finetune.history['accuracy']
    total_val_accuracy = history.history['val_accuracy'] + history_finetune.history['val_accuracy']

    ax1.plot(total_accuracy, label='Train Accuracy', color='#1f77b4', linewidth=2)
    ax1.plot(total_val_accuracy, label='Val Accuracy', color='#ff7f0e', linewidth=2)
    # Linea verticale per indicare l'inizio del fine-tuning
    ax1.axvline(x=split_point - 1, color='gray', linestyle='--', alpha=0.7, label='Fine-Tuning Start')

    ax1.set_title('Model Accuracy', fontsize=14)
    ax1.set_xlabel('Epochs', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.legend(loc='lower right')
    ax1.grid(True, linestyle=':', alpha=0.6)

    # --- 2. GRAFICO LOSS ---
    # Uniamo i dati delle due fasi
    total_loss = history.history['loss'] + history_finetune.history['loss']
    total_val_loss = history.history['val_loss'] + history_finetune.history['val_loss']

    ax2.plot(total_loss, label='Train Loss', color='#d62728', linewidth=2)
    ax2.plot(total_val_loss, label='Val Loss', color='#2ca02c', linewidth=2)
    # Linea verticale per indicare l'inizio del fine-tuning
    ax2.axvline(x=split_point - 1, color='gray', linestyle='--', alpha=0.7, label='Fine-Tuning Start')

    ax2.set_title('Model Loss', fontsize=14)
    ax2.set_xlabel('Epochs', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.legend(loc='upper right')
    ax2.grid(True, linestyle=':', alpha=0.6)

    # --- SALVATAGGIO DELLA FIGURA ---
    # Assicurati che la cartella esista
    os.makedirs(ECG_METRICHE, exist_ok=True)

    plot_filename = f'metrics_plot_fold_{fold}.png'
    plot_save_path = os.path.join(ECG_METRICHE, plot_filename)

    # tight_layout evita che le scritte si sovrappongano
    plt.tight_layout()
    plt.savefig(plot_save_path, dpi=300) # 300 DPI per un'alta definizione da Tesi
    plt.close()

    print(f"Grafico delle metriche salvato per il fold {fold} in: {plot_save_path}")

